# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset ID: {metadata.id}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their fields by @id
recordsets = list(metadata.record_sets)
if len(recordsets) == 0:
    print("No record sets found in metadata.\nFor some Croissant datasets, top-level Files are the only record sets.")
    # List dataset files if present
    files = list(metadata.files)
    if files:
        print("Available dataset files (as RecordSets):")
        for f in files:
            print(f"  - File: {f.id} (name: {f.name})")
        # We can treat each File as a record set for mlcroissant
        file_recordsets = [f.id for f in files]
        recordsets = file_recordsets
    else:
        print("No files found either. Please check schema.")
else:
    print("Record Sets found in metadata:")
    for rs in recordsets:
        print(f"- RecordSet @id: {rs.id}, name: {rs.name}")

# For each file/record set, print the fields/columns/headers
for record_set_id in recordsets:
    print(f"\nFields/columns for RecordSet/File @id: {record_set_id}")
    try:
        # Try to get the RecordSet (or FileObject) and list its fields
        record_set_obj = metadata.get(record_set_id)
        if hasattr(record_set_obj, 'fields') and record_set_obj.fields:
            for fld in record_set_obj.fields:
                print(f"  - Field @id: {fld.id}, name: {fld.name}, type: {fld.data_type}")
        elif hasattr(record_set_obj, 'columns') and record_set_obj.columns:
            for col in record_set_obj.columns:
                print(f"  - Column @id: {col.id}, name: {col.name}, type: {getattr(col, 'data_type', None)}")
        else:
            print("  No fields or columns defined in this record set. Data schema may be inferred on load.")
    except Exception as e:
        print(f"  Could not inspect fields for record set {record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set (by `@id`) into a DataFrame for analysis.

In [ ]:
#
# Getting all available record set @ids for extraction
#
# If the dataset has no explicit record sets, we use file @ids as record_set ids.
record_set_ids = recordsets  # Identified above
print("Using record set @ids for extraction:")
for rid in record_set_ids:
    print(f"- {rid}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set/file @id: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load data for record set {record_set_id}: {e}")

# Select one record set id for further analysis
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nSelected main record set for analysis: {main_record_set_id}")
    df_main = dataframes[main_record_set_id]
    print("Sample data:")
    display_cols = list(df_main.columns[:8])
    print(df_main[display_cols].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select a numeric field @id for analysis. We'll inspect the main DataFrame for candidates.

df = df_main.copy()
df_numeric = df.select_dtypes(include='number')
print("Numeric columns found:", df_numeric.columns.tolist())

# Choose a numeric field for exploration (e.g. abundance, mw, or peptide_count if present)
# You can change the below to your desired field
if len(df_numeric.columns) > 0:
    numeric_field = df_numeric.columns[0]
else:
    print("No numeric columns available for EDA.")
    numeric_field = None

if numeric_field:
    # Example: filter out entries with value greater than a threshold
    threshold = df[numeric_field].mean()  # or a fixed number
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Try grouping by a common text/categorical field (e.g. 'Description' or 'Protein')
    potential_group_fields = [col for col in df.columns if (df[col].dtype == 'object' and col.lower() != numeric_field.lower())]
    if len(potential_group_fields) > 0:
        group_field = potential_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (showing means for filtered records):")
        print(grouped_df.head())
    else:
        print("No suitable group/categorical field found for grouping.")
else:
    print("Skipping EDA steps as no numeric column found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # Optionally, boxplot by group_field
    if 'group_field' in locals() and group_field:
        top_groups = filtered_df[group_field].value_counts().index[:5].tolist()
        filtered_plot_df = filtered_df[filtered_df[group_field].isin(top_groups)]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_plot_df)
        plt.title(f'{numeric_field} across {group_field} (top 5 categories)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 dataset on Mass Spectrometry Analysis of Extracellular Vesicles from its Croissant schema with `mlcroissant`, explored its structure using `@id` references, and performed basic exploratory data analysis and visualization. Further analysis can focus on filtering proteins by abundance, searching for specific post-translational modifications, or cross-analyzing peptide data by sample groups.

_Modify and extend this notebook as needed for downstream machine learning or biological data analysis tasks._